## Photos ##

### Cleaning 06/16/2026 ###

* Goal 1: Check all 9,346 images and flag corrupted, blank, or wrong size (not 224 x 224), or wrong format.
* Goal 2: Check all images' correctness by removing wrong breeds or species based on observations from 06/11/2026

Below is the script I ran to check for Goal 1 tasks. Looks like there are no corupted, blank or wrong sized images.

In [2]:
import os
from PIL import Image

def verify_images_in_dataset(base_dirs):
    """
    Recursively verifies .jpg files and reports a summary of all issues found.
    """
    report = {
        "size_mismatch": [],
        "corrupted": [],
        "total_checked": 0
    }
    
    for base_dir in base_dirs:
        if not os.path.exists(base_dir):
            print(f"Directory not found: {base_dir}")
            continue
            
        print(f"Processing directory: {base_dir}...")
        
        for root, dirs, files in os.walk(base_dir):
            for file in files:
                if file.lower().endswith(".jpg"):
                    report["total_checked"] += 1
                    filepath = os.path.join(root, file)
                    try:
                        with Image.open(filepath) as img:
                            # Verify dimensions
                            if img.size != (224, 224):
                                report["size_mismatch"].append(f"{filepath} (Size: {img.size})")
                            
                            # Verify integrity
                            img.verify()
                    except Exception as e:
                        report["corrupted"].append(f"{filepath} (Error: {e})")
        
    # Print the summary report
    print("\n--- Verification Summary ---")
    print(f"Total files checked: {report['total_checked']}")
    
    if report["size_mismatch"]:
        print(f"\nIssues found: {len(report['size_mismatch'])} files with wrong dimensions:")
        for item in report["size_mismatch"]:
            print(f" - {item}")
            
    if report["corrupted"]:
        print(f"\nIssues found: {len(report['corrupted'])} corrupted or invalid files:")
        for item in report["corrupted"]:
            print(f" - {item}")
            
    if not report["size_mismatch"] and not report["corrupted"]:
        print("No issues found. All images are valid and sized correctly.")

# Define your dataset paths
dataset_paths = ["../data/test", "../data/train", "../data/valid"]

# Run the verification
verify_images_in_dataset(dataset_paths)

Processing directory: ../data/test...
Processing directory: ../data/train...
Processing directory: ../data/valid...

--- Verification Summary ---
Total files checked: 10046
No issues found. All images are valid and sized correctly.


To accompilsh Goal 2, I manually removed:
* test
    * "Coyote" folder
    * 01.jpg from "Bulldog" folder; French Bulldog
     * 04.jpg, 05.jpg, 06.jpg, 07.jpg, 08.jpg, 09.jpg, and 10.jpg from "Bulldog" folder; Boston Terriers
* train
    * "Coyote" folder
    * 003.jpg, 004.jpg, 005.jpg, 006.jpg, and 007.jpg, 034.jpg, 045.jpg, 0.61.jpg, 096.jpg; Pitbulls, American Bulldog
    * 017.jpg from "Bulldog" folder; French Bulldog
    * 044.jpg from "Bulldog" folder; German Shepherd
    * 046.jpg, 047.jpg, 048.jpg, 049.jpg, 050.jpg, 051.jpg, 052.jpg, 053.jpg, 054.jpg, 055.jpg, 056.jpg, 057.jpg, 058.jpg, 059.jpg, 060.jpg, 062.jpg, 0.63.jpg, 064.jpg, 065.jpg, 066.jpg, 067.jpg, 068.jpg, 069.jpg, 070.jpg, 071.jpg, 072.jpg, 073.jpg, 074.jpg, 075.jpg, 076.jpg, 077.jpg, 078.jpg, 079.jpg, 080.jpg, 081.jpg, 082.jpg, 084.jpg, 085.jpg, 086.jpg, 087.jpg, 088.jpg, 089.jpg, 090.jpg, 091.jpg, 092.jpg, 093.jpg, 094.jpg, 095.jpg, 097.jpg, 098.jpg, 099.jpg, 100.jpg, 101.jpg, 102.jpg, 103.jpg, 104.jpg, 105.jpg, 106.jpg, 107.jpg, 108.jpg, 109.jpg, 110.jpg, 111.jpg, 112.jpg, 113.jpg, 114.jpg, 115.jpg, 116.jpg, 117.jpg, 118.jpg, 119.jpg, 120.jpg, 121.jpg, 122.jpg, 123.jpg, 124.jpg, 125.jpg from "Bulldog" folder; Boston Terriers
* valid 
    * "Coyote" folder
    * 05.jpg, 06.jpg, 08.jpg, from "Bulldog" folder; Boston Terrier


Note that this leaves "Bulldog" folder of test folder with only 2 images.



Updated the dogs.csv using the script below to create dogs_updated.csv based on the changes above.

In [3]:
import pandas as pd
from pathlib import Path

# Paths
input_path = Path("../data/dogs.csv")
output_path = Path("../data/dogs_updated.csv")

# Load CSV
df = pd.read_csv(input_path)

# Helper to drop by pattern
def drop_pattern(df, dataset, label=None, folder=None, filenames=None):
    """
    Returns a new df with rows removed that match:
    - data set == dataset
    - labels == label (if provided)
    - filepaths contains /folder/ (if provided)
    - filename in filenames (if provided)
    """
    mask = df["data set"].eq(dataset)
    if label is not None:
        mask &= df["labels"].eq(label)
    if folder is not None:
        mask &= df["filepaths"].str.contains(f"/{folder}/", regex=False)
    if filenames is not None:
        file_names_only = df["filepaths"].str.extract(r"([^/]+)$")[0]
        mask &= file_names_only.isin(filenames)
    return df[~mask]

# --- 1) Remove all Coyote folders across datasets ---
df = df[~df["filepaths"].str.contains("/Coyote/", regex=False)]

# --- 2) TEST/Bulldog specific files ---

# 01.jpg from "Bulldog" folder; French Bulldog
df = drop_pattern(
    df,
    dataset="test",
    label="Bulldog",
    folder="Bulldog",
    filenames=["01.jpg"],
)

# 04.jpg–10.jpg from "Bulldog" folder; Boston Terriers
boston_test_files = [f"{i:02d}.jpg" for i in range(4, 11)]  # 04.jpg–10.jpg
df = drop_pattern(
    df,
    dataset="test",
    label="Bulldog",
    folder="Bulldog",
    filenames=boston_test_files,
)

# --- 3) TRAIN removals ---

# Pitbulls, American Bulldog in train: specific filenames
train_pit_files = [
    "003.jpg", "004.jpg", "005.jpg", "006.jpg", "007.jpg",
    "034.jpg", "045.jpg", "061.jpg", "096.jpg",
]
file_names_only = df["filepaths"].str.extract(r"([^/]+)$")[0]
mask_train_pit = (
    df["data set"].eq("train") &
    file_names_only.isin(train_pit_files)
)
df = df[~mask_train_pit]

# 017.jpg from "Bulldog" folder; French Bulldog (train)
df = drop_pattern(
    df,
    dataset="train",
    label="Bulldog",
    folder="Bulldog",
    filenames=["017.jpg"],
)

# 044.jpg from "Bulldog" folder; German Shepherd (train)
df = drop_pattern(
    df,
    dataset="train",
    label="Bulldog",
    folder="Bulldog",
    filenames=["044.jpg"],
)

# Long list (Boston Terriers) in train/Bulldog
nums = (
    list(range(46, 61))  # 046–060
    + [62, 63]
    + list(range(64, 77))  # 064–076
    + list(range(77, 83))  # 077–082
    + list(range(84, 96))  # 084–095
    + [97, 98, 99]
    + list(range(100, 126))  # 100–125
)
train_boston_files = [f"{n:03d}.jpg" for n in nums]

df = drop_pattern(
    df,
    dataset="train",
    label="Bulldog",
    folder="Bulldog",
    filenames=train_boston_files,
)

# --- 4) VALID removals ---

# 05.jpg, 06.jpg, 08.jpg from "Bulldog" folder; Boston Terrier (valid)
valid_boston_files = ["05.jpg", "06.jpg", "08.jpg"]
df = drop_pattern(
    df,
    dataset="valid",
    label="Bulldog",
    folder="Bulldog",
    filenames=valid_boston_files,
)

# --- Save updated CSV ---
df.to_csv(output_path, index=False)

print(f"Saved updated CSV to: {output_path}")
print(f"Remaining rows: {len(df)}")

# Optional sanity check: Bulldog in test set
bulldog_test = df[(df["data set"] == "test") & (df["labels"] == "Bulldog")]
print("Bulldog/test count:", len(bulldog_test))
print(bulldog_test.head())

Saved updated CSV to: ../data/dogs_updated.csv
Remaining rows: 8694
Bulldog/test count: 2
                filepaths   labels data set
8147  test/Bulldog/02.jpg  Bulldog     test
8148  test/Bulldog/03.jpg  Bulldog     test


### Observations 06/11/2026 ###
* A "Bulldog" as a breed refers to the "English Bulldog." The test data includes images of English Bulldogs, French Bulldogs, and Boston Terriers. Given that there are already "French Bulldog" and "Boston Terrier" as  separate breeds, this could be confusing for the algorithm. 
* A "Coyote" is not a dog.

In [2]:
from IPython.display import Image, display


In [14]:
display(Image(url="../data/test/Afghan/01.jpg", width=400))


In [15]:
display(Image(url="../data/test/African Wild Dog/01.jpg", width=400))


In [6]:
display(Image(url="../data/test/Airedale/01.jpg", width=400))

In [7]:
display(Image(url="../data/test/American Hairless/01.jpg", width=400))

In [8]:
display(Image(url="../data/test/American Spaniel/01.jpg", width=400))

In [9]:
display(Image(url="../data/test/Basenji/01.jpg", width=400))

In [10]:
display(Image(url="../data/test/Basset/01.jpg", width=400))

In [11]:
display(Image(url="../data/test/Beagle/01.jpg", width=400))

In [12]:
display(Image(url="../data/test/Bearded Collie/01.jpg", width=400))

In [13]:
display(Image(url="../data/test/Bermaise/01.jpg", width=400))

In [16]:
display(Image(url="../data/test/Bichon Frise/01.jpg", width=400))


In [17]:
display(Image(url="../data/test/Blenheim/01.jpg", width=400))

In [18]:
display(Image(url="../data/test/Bloodhound/01.jpg", width=400))

In [19]:
display(Image(url="../data/test/Bluetick/01.jpg", width=400))

In [20]:
display(Image(url="../data/test/Border Collie/01.jpg", width=400))

In [21]:
display(Image(url="../data/test/Borzoi/01.jpg", width=400))

In [22]:
display(Image(url="../data/test/Boston Terrier/01.jpg", width=400))

In [23]:
display(Image(url="../data/test/Boxer/01.jpg", width=400))

In [24]:
display(Image(url="../data/test/Bull Mastiff/01.jpg", width=400))

In [25]:
display(Image(url="../data/test/Bull Terrier/01.jpg", width=400))

In [26]:
display(Image(url="../data/test/Bulldog/01.jpg", width=400))

In [27]:
display(Image(url="../data/test/Cairn/01.jpg", width=400))

In [28]:
display(Image(url="../data/test/Chihuahua/01.jpg", width=400))


In [29]:
display(Image(url="../data/test/Chinese Crested/01.jpg", width=400))

In [30]:
display(Image(url="../data/test/Chow/01.jpg", width=400))

In [31]:
display(Image(url="../data/test/Clumber/01.jpg", width=400))


In [32]:
display(Image(url="../data/test/Cockapoo/01.jpg", width=400))

In [33]:
display(Image(url="../data/test/Cocker/01.jpg", width=400))

In [34]:
display(Image(url="../data/test/Collie/01.jpg", width=400))

In [35]:
display(Image(url="../data/test/Corgi/01.jpg", width=400))

In [36]:
display(Image(url="../data/test/Coyote/01.jpg", width=400))

In [37]:
display(Image(url="../data/test/Dalmation/01.jpg", width=400))

In [38]:
display(Image(url="../data/test/Dhole/01.jpg", width=400))

In [39]:
display(Image(url="../data/test/Dingo/01.jpg", width=400))

In [40]:
display(Image(url="../data/test/Doberman/01.jpg", width=400))

In [41]:
display(Image(url="../data/test/Elk Hound/01.jpg", width=400))

In [42]:
display(Image(url="../data/test/French Bulldog/01.jpg", width=400))

In [43]:
display(Image(url="../data/test/German Sheperd/01.jpg", width=400))

In [44]:
display(Image(url="../data/test/Golden Retriever/01.jpg", width=400))

In [45]:
display(Image(url="../data/test/Great Dane/01.jpg", width=400))

In [46]:
display(Image(url="../data/test/Great Perenees/01.jpg", width=400))


In [3]:
display(Image(url="../data/test/Greyhound/01.jpg", width=400))

In [4]:
display(Image(url="../data/test/Groenendael/01.jpg", width=400))

In [5]:
display(Image(url="../data/test/Irish Spaniel/01.jpg", width=400))

In [6]:
display(Image(url="../data/test/Irish Wolfhound/01.jpg", width=400))

In [7]:
display(Image(url="../data/test/Japanese Spaniel/01.jpg", width=400))

In [8]:
display(Image(url="../data/test/Komondor/01.jpg", width=400))

In [9]:
display(Image(url="../data/test/Labradoodle/01.jpg", width=400))

In [10]:
display(Image(url="../data/test/Labrador/01.jpg", width=400))

In [11]:
display(Image(url="../data/test/Lhasa/01.jpg", width=400))

In [12]:
display(Image(url="../data/test/Malinois/01.jpg", width=400))

In [13]:
display(Image(url="../data/test/Maltese/01.jpg", width=400))

In [14]:
display(Image(url="../data/test/Mex Hairless/01.jpg", width=400))

In [15]:
display(Image(url="../data/test/Newfoundland/01.jpg", width=400))

In [16]:
display(Image(url="../data/test/Pekinese/01.jpg", width=400))

In [17]:
display(Image(url="../data/test/Pit Bull/01.jpg", width=400))

In [18]:
display(Image(url="../data/test/Pomeranian/01.jpg", width=400))

In [19]:
display(Image(url="../data/test/Poodle/01.jpg", width=400))

In [20]:
display(Image(url="../data/test/Pug/01.jpg", width=400))

In [21]:
display(Image(url="../data/test/Rhodesian/01.jpg", width=400))

In [22]:
display(Image(url="../data/test/Rottweiler/01.jpg", width=400))

In [23]:
display(Image(url="../data/test/Saint Bernard/01.jpg", width=400))

In [24]:
display(Image(url="../data/test/Schnauzer/01.jpg", width=400))

In [25]:
display(Image(url="../data/test/Scotch Terrier/01.jpg", width=400))

In [26]:
display(Image(url="../data/test/Shar_Pei/01.jpg", width=400))

In [27]:
display(Image(url="../data/test/Shiba Inu/01.jpg", width=400))

In [28]:
display(Image(url="../data/test/Shih-Tzu/01.jpg", width=400))

In [29]:
display(Image(url="../data/test/Siberian Husky/01.jpg", width=400))

In [30]:
display(Image(url="../data/test/Vizsla/01.jpg", width=400))

In [31]:
display(Image(url="../data/test/Yorkie/01.jpg", width=400))